In [12]:
import subprocess, sys
try:
    import torch_geometric
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "torch-geometric", "scikit-learn", "-q"], check=False)
    import torch_geometric
print(f"torch-geometric: {torch_geometric.__version__}")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 39.1 MB/s eta 0:00:00
torch-geometric: 2.8.0.post1


Graph Feature Construction

Converts formatted transaction CSV into PyTorch Geometric graph tensors.


1. Load Formatted Transactions


In [4]:
import os, glob
import pandas as pd
import numpy as np
import torch

LOCAL_PATH = '/home/shreyas-nalle/Desktop/Delusional/model/formatted_transactions.csv'
paths_to_check = [LOCAL_PATH, 'model/formatted_transactions.csv', 'formatted_transactions.csv']
kaggle_glob = glob.glob('/kaggle/input/**/formatted_transactions.csv', recursive=True)
if kaggle_glob: paths_to_check.insert(0, kaggle_glob[0])

csv_path = next((p for p in paths_to_check if os.path.exists(p)), None)
if csv_path is None:
    raise FileNotFoundError('formatted_transactions.csv not found!')

df_edges = pd.read_csv(csv_path)
print(f'Loaded {len(df_edges):,} transactions from {csv_path}')


Loaded 5,000,000 transactions from /kaggle/input/datasets/shreyasnalle/formatted-transactions/formatted_transactions.csv


2. Check Timestamp Range


In [5]:
df_edges["Timestamp"] = df_edges["Timestamp"] - df_edges["Timestamp"].min()
n_days = int(df_edges["Timestamp"].max() / (3600 * 24) + 1)
n_samples = len(df_edges)
timestamp_range = df_edges["Timestamp"].max()
print(f"Timestamp range: {timestamp_range:,} seconds ({n_days} days, {n_samples:,} transactions)")


Timestamp range: 5,938,140 seconds (69 days, 5,000,000 transactions)


3. Create Node Feature Matrix (x)


In [6]:
max_n_id = int(df_edges[["from_id", "to_id"]].to_numpy().max()) + 1
x = torch.ones((max_n_id, 1), dtype=torch.float)
print(f"Total unique account nodes: {max_n_id:,}, Node feature matrix x: {x.shape}")


Total unique account nodes: 1,754,264, Node feature matrix x: torch.Size([1754264, 1])


4. Create Edge Index Tensor (edge_index)


In [7]:
edge_index = torch.LongTensor(df_edges[["from_id", "to_id"]].to_numpy().T)
print(f"Edge index shape: {edge_index.shape}")


Edge index shape: torch.Size([2, 5000000])


5. Create Edge Feature Tensor (edge_attr)


In [8]:
curr_col = "Received Currency" if "Received Currency" in df_edges.columns else "Receiving Currency"
edge_features = ["Timestamp", "Amount Received", curr_col, "Payment Format"]
edge_attr = torch.tensor(df_edges[edge_features].to_numpy()).float()
print(f"Edge attribute matrix shape: {edge_attr.shape}")


Edge attribute matrix shape: torch.Size([5000000, 4])


6. Create Timestamps and Label Tensor (y)


In [9]:
timestamps = torch.FloatTensor(df_edges["Timestamp"].to_numpy())
y = torch.LongTensor(df_edges["Is Laundering"].to_numpy())
print(f"Labels tensor shape: {y.shape}, Illicit ratio: {y.float().mean()*100:.3f}%")


Labels tensor shape: torch.Size([5000000]), Illicit ratio: 0.056%


6.5 Add Multi-GNN Edge Features (Ports & Time-Deltas)


In [10]:
import numpy as np

def compute_ports(edge_idx, timestamps, num_nodes):
    ports = torch.zeros(edge_idx.shape[1], 1)
    adj_list = {i: [] for i in range(num_nodes)}
    edges = torch.cat((edge_idx.T, timestamps.reshape(-1, 1)), dim=1)
    for u, v, t in edges:
        adj_list[int(v)].append((int(u), int(t)))
    ports_dict = {}
    for v, nbs in adj_list.items():
        if len(nbs) < 1: continue
        a = np.array(nbs)
        a = a[a[:, -1].argsort()]
        _, idx = np.unique(a[:, 0], return_index=True)
        nbs_unique = a[np.sort(idx)][:, 0]
        for i, u in enumerate(nbs_unique):
            ports_dict[(u, v)] = i
    for i, e in enumerate(edge_idx.T):
        ports[i] = ports_dict.get(tuple(e.numpy()), 0)
    return ports

def compute_time_deltas(edge_idx, timestamps, num_nodes):
    tds = torch.zeros(edge_idx.shape[1], 1)
    adj_edges = {i: [] for i in range(num_nodes)}
    edges = torch.cat((edge_idx.T, timestamps.reshape(-1, 1)), dim=1)
    for i, (u, v, t) in enumerate(edges):
        adj_edges[int(v)].append((i, int(u), int(t)))
    for v, nbs in adj_edges.items():
        if len(nbs) < 1: continue
        a = np.array(nbs)
        a = a[a[:, -1].argsort()]
        a_tds = [0] + [a[i+1, -1] - a[i, -1] for i in range(a.shape[0]-1)]
        for idx_e, td in zip(a[:, 0], a_tds):
            tds[int(idx_e)] = td
    return tds

print("Computing incoming/outgoing ports and time-deltas (this may take a minute)...")
in_ports  = compute_ports(edge_index, timestamps, max_n_id)
out_ports = compute_ports(edge_index.flipud(), timestamps, max_n_id)
in_tds    = compute_time_deltas(edge_index, timestamps, max_n_id)
out_tds   = compute_time_deltas(edge_index.flipud(), timestamps, max_n_id)

edge_attr = torch.cat([edge_attr, in_ports, out_ports, in_tds, out_tds], dim=1)
print(f"New edge_attr shape (w/ ports & tds): {edge_attr.shape}")


Computing incoming/outgoing ports and time-deltas (this may take a minute)...
New edge_attr shape (w/ ports & tds): torch.Size([5000000, 8])


7. Wrap into Data Object


In [13]:
from torch_geometric.data import Data

graph_data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)
graph_data.timestamps = timestamps
print(f"Graph Data object created with {graph_data.num_nodes:,} nodes and {graph_data.num_edges:,} edges")


Graph Data object created with 1,754,264 nodes and 5,000,000 edges


8. Save Graph Data Tensors


In [14]:
save_path = "graph_data.pt"
torch.save({"x": x, "edge_index": edge_index, "edge_attr": edge_attr, "timestamps": timestamps, "y": y}, save_path)
print(f"Saved graph_data.pt -> {save_path} ({os.path.getsize(save_path) / 1e9:.2f} GB)")


Saved graph_data.pt -> graph_data.pt (0.31 GB)
